# EXP-7 — ConvNeXt-Tiny: Moderate Class-Balanced Fine-tuning + Regularization

**Mục tiêu:** tiếp tục trực tiếp từ checkpoint tốt nhất của EXP-6, không đổi backbone, tập trung xử lý **bias về FAKE + overfitting/variance**.

EXP-7 chỉ dùng:
- `train/` để cập nhật trọng số;
- `val/` để đánh giá, early stopping, chọn checkpoint và threshold;
- **KHÔNG đọc, KHÔNG load, KHÔNG evaluate `test/`.**

EXP-6 có Validation Accuracy khoảng 0.876 và confusion matrix cho thấy REAL bị dự đoán thành FAKE nhiều. Vì vậy EXP-7 ưu tiên cải thiện REAL recall/accuracy thay vì tiếp tục tăng learning rate.

In [1]:
# =========================
# 1. Install / imports
# =========================
!pip install -q torch torchvision torchaudio pandas numpy matplotlib seaborn scikit-learn tqdm

import os, gc, json, time, copy, random, shutil, zipfile, io, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

ImageFile.LOAD_TRUNCATED_IMAGES = True
print("PyTorch:", torch.__version__)


PyTorch: 2.11.0+cu128


In [2]:
# =========================
# 2. Reproducibility + device
# =========================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB")


Device: cuda
GPU: Tesla T4
GPU memory: 14.56 GB


In [3]:
# =========================
# 3. Google Drive + paths
# =========================
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive"
ZIP_PATH = os.path.join(DRIVE_ROOT, "face_frames.zip")
EXTRACT_DIR = "/content/faceforensics"

EXPERIMENT_DIR = os.path.join(DRIVE_ROOT, "FaceForensics_Experiments")
FINETUNE_DIR = os.path.join(EXPERIMENT_DIR, "finetune_convnext_tiny")
CHECKPOINT_DIR = os.path.join(FINETUNE_DIR, "checkpoints")
HISTORY_DIR = os.path.join(FINETUNE_DIR, "histories")
RESULT_DIR = os.path.join(FINETUNE_DIR, "results")
BEST_TRACKER_PATH = os.path.join(FINETUNE_DIR, "best_tracker.json")

for d in [EXPERIMENT_DIR, FINETUNE_DIR, CHECKPOINT_DIR, HISTORY_DIR, RESULT_DIR]:
    os.makedirs(d, exist_ok=True)

print("ZIP:", ZIP_PATH)
print("Extract:", EXTRACT_DIR)
print("Best tracker:", BEST_TRACKER_PATH)


Mounted at /content/drive
ZIP: /content/drive/MyDrive/face_frames.zip
Extract: /content/faceforensics
Best tracker: /content/drive/MyDrive/FaceForensics_Experiments/finetune_convnext_tiny/best_tracker.json


In [4]:
# =========================
# 4. Extract dataset if needed
# =========================
if not os.path.exists(EXTRACT_DIR):
    print("Extracting dataset...")
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
    print(f"Done in {time.time()-t0:.1f}s")
else:
    print("Dataset already extracted; skip.")


Extracting dataset...
Done in 37.3s


In [5]:
# =========================
# 5. Locate train / val only
# =========================
def find_dataset_root(base_dir):
    candidates = []
    for root, dirs, files in os.walk(base_dir):
        names = {d.lower() for d in dirs}
        if {"train", "val"}.issubset(names):
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError("Không tìm thấy thư mục train/val.")
    return candidates[0]

DATA_DIR = find_dataset_root(EXTRACT_DIR)
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")

print(DATA_DIR)
print(TRAIN_DIR)
print(VAL_DIR)


/content/faceforensics/face_frames
/content/faceforensics/face_frames/train
/content/faceforensics/face_frames/val


## EXP-7 — Hypothesis

EXP-6 đã có augmentation và gentle full fine-tuning. EXP-7 chỉ thay đổi các yếu tố có liên quan trực tiếp đến kết quả hiện tại:

1. **Moderate class-weighted loss:** REAL = 1.5, FAKE = 1.0 để giảm xu hướng dự đoán FAKE.
2. **Dropout:** 0.25 → 0.35 để giảm variance.
3. **Weight decay:** 1e-4 → 3e-4 để regularize mạnh hơn vừa phải.
4. **Label smoothing:** 0.05 → 0.03 để không làm mất tín hiệu phân biệt.
5. Augmentation giảm blur/noise/rotation nhưng tăng nhẹ JPEG robustness.
6. Giữ ConvNeXt-Tiny, GELU và LayerNorm; không thay activation/backbone normalization.
7. Giữ batch size 64, AMP, channels-last và gradient clipping.

**Không dùng WeightedRandomSampler**, vì class weight đã xử lý imbalance; dùng cả hai sẽ cân bằng quá mạnh.

In [6]:
# =========================
# 6. EXP-7 Configuration
# =========================
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 10
EARLY_STOPPING_PATIENCE = 4

# Continue gently from BEST EXP-6.
LR_STAGE1 = 1e-6
LR_STAGE2 = 2e-6
LR_STAGE3 = 5e-6
LR_STAGE4_HEAD = 1e-5

# Regularization: moderately stronger than EXP-6.
WEIGHT_DECAY = 3e-4
LABEL_SMOOTHING = 0.03
DROPOUT_P = 0.35
GRAD_CLIP_NORM = 1.0

# Moderate class weighting — NOT inverse-frequency weighting.
REAL_CLASS_WEIGHT = 1.5
FAKE_CLASS_WEIGHT = 1.0

# Moderate augmentation.
JPEG_P = 0.50
JPEG_QUALITY_RANGE = (40, 70)
BLUR_P = 0.15
NOISE_P = 0.10
COLOR_P = 0.30
ROTATION_DEG = 5

NUM_WORKERS = 2
PIN_MEMORY = device.type == "cuda"
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2
USE_AMP = device.type == "cuda"

EXP_NAME = "exp7_moderate_classbalanced_reg"
MODEL_NAME = "convnext_tiny"

print({
    "batch": BATCH_SIZE,
    "epochs": EPOCHS,
    "LRs": [LR_STAGE1, LR_STAGE2, LR_STAGE3, LR_STAGE4_HEAD],
    "REAL_weight": REAL_CLASS_WEIGHT,
    "FAKE_weight": FAKE_CLASS_WEIGHT,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "dropout": DROPOUT_P,
    "JPEG_p": JPEG_P,
    "blur_p": BLUR_P,
    "noise_p": NOISE_P,
    "rotation": ROTATION_DEG,
    "TEST_USED": False
})

{'batch': 64, 'epochs': 10, 'LRs': [1e-06, 2e-06, 5e-06, 1e-05], 'REAL_weight': 1.5, 'FAKE_weight': 1.0, 'weight_decay': 0.0003, 'label_smoothing': 0.03, 'dropout': 0.35, 'JPEG_p': 0.5, 'blur_p': 0.15, 'noise_p': 0.1, 'rotation': 5, 'TEST_USED': False}


In [7]:
# =========================
# 7. Dataset
# =========================
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CLASS_TO_LABEL = {"REAL": 0, "FAKE": 1}
LABEL_TO_CLASS = {0: "REAL", 1: "FAKE"}

def create_dataframe(split_dir):
    records = []
    for class_dir in Path(split_dir).iterdir():
        if not class_dir.is_dir():
            continue
        class_name = class_dir.name.upper()
        if class_name not in CLASS_TO_LABEL:
            continue
        label = CLASS_TO_LABEL[class_name]
        for p in class_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
                records.append({
                    "image_path": str(p),
                    "label": label,
                    "class_name": class_name
                })
    return pd.DataFrame(records)

train_df = create_dataframe(TRAIN_DIR)
val_df = create_dataframe(VAL_DIR)

print("TRAIN:", len(train_df))
print(train_df["class_name"].value_counts())
print("\nVAL:", len(val_df))
print(val_df["class_name"].value_counts())


TRAIN: 54346
class_name
FAKE    46448
REAL     7898
Name: count, dtype: int64

VAL: 7002
class_name
FAKE    5982
REAL    1020
Name: count, dtype: int64


In [8]:
# =========================
# 8. EXP-7 moderate augmentation
# =========================
class RandomJPEGCompression:
    def __init__(self, p=0.50, quality_range=(40, 70)):
        self.p = p
        self.quality_range = quality_range

    def __call__(self, img):
        if random.random() > self.p:
            return img
        q = random.randint(*self.quality_range)
        buffer = io.BytesIO()
        img.convert("RGB").save(buffer, format="JPEG", quality=q)
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")

class RandomGaussianNoise:
    def __init__(self, p=0.10, sigma_range=(0.01, 0.025)):
        self.p = p
        self.sigma_range = sigma_range

    def __call__(self, tensor):
        if random.random() > self.p:
            return tensor
        sigma = random.uniform(*self.sigma_range)
        noise = torch.randn_like(tensor) * sigma
        return torch.clamp(tensor + noise, 0.0, 1.0)

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(ROTATION_DEG),
    transforms.ColorJitter(
        brightness=0.10, contrast=0.10,
        saturation=0.08, hue=0.02
    ),
    RandomJPEGCompression(JPEG_P, JPEG_QUALITY_RANGE),
    transforms.ToTensor(),
    RandomGaussianNoise(NOISE_P, (0.01, 0.025)),
    transforms.RandomApply(
        [transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.8))],
        p=BLUR_P
    ),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

print("EXP-7 augmentation ready. Validation augmentation = NONE")

EXP-7 augmentation ready. Validation augmentation = NONE


In [9]:
# =========================
# 9. DataLoader — mini-batch + vectorized tensor batches
# =========================
class FaceForensicsDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        try:
            img = Image.open(row["image_path"]).convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
            return img, int(row["label"])
        except Exception as e:
            raise RuntimeError(f"Cannot read {row['image_path']}: {e}")

train_dataset = FaceForensicsDataset(train_df, train_transform)
val_dataset = FaceForensicsDataset(val_df, eval_transform)

loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


Train batches: 850
Val batches: 110


In [10]:
# =========================
# 10. Load BEST EXP-6 checkpoint ONLY
# =========================
# Do not use best_tracker.json here because it can point to an older EXP.
# EXP-7 must start explicitly from EXP-6.
EXP6_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "convnext_tiny_exp6_aug_gentle_fullft_best.pth"
)

if not os.path.exists(EXP6_CHECKPOINT):
    raise FileNotFoundError(
        "Không tìm thấy BEST EXP-6 checkpoint: " + EXP6_CHECKPOINT
    )

print("Source experiment: EXP-6")
print("Checkpoint:", EXP6_CHECKPOINT)
print("TEST_USED: False")

Source experiment: EXP-6
Checkpoint: /content/drive/MyDrive/FaceForensics_Experiments/finetune_convnext_tiny/checkpoints/convnext_tiny_exp6_aug_gentle_fullft_best.pth
TEST_USED: False


In [11]:
# =========================
# 11. ConvNeXt-Tiny model + safe classifier head
# =========================
class ConvNeXtHead(nn.Module):
    def __init__(self, in_features, dropout=0.20, use_batchnorm=False):
        super().__init__()
        self.bn = nn.BatchNorm1d(in_features) if use_batchnorm else nn.Identity()
        self.dropout = nn.Dropout(dropout)
        # Keep the original 768 -> 2 classifier shape so the previous classifier
        # can be reused instead of throwing away its learned decision boundary.
        self.fc = nn.Linear(in_features, 2, bias=True)

    def forward(self, x):
        x = self.bn(x)
        x = self.dropout(x)
        return self.fc(x)

def build_model():
    model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
    in_features = model.classifier[-1].in_features

    model.classifier = nn.Sequential(
        model.classifier[0],  # LayerNorm2d
        model.classifier[1],  # Flatten
        ConvNeXtHead(
            in_features,
            dropout=DROPOUT_P,
            use_batchnorm=FALSE
        )
    )
    return model

model = build_model()

checkpoint = torch.load(
    EXP6_CHECKPOINT, map_location="cpu", weights_only=False
)
state = checkpoint.get("model_state_dict", checkpoint)

# Load all backbone weights.
backbone_state = {
    k: v for k, v in state.items()
    if not k.startswith("classifier.")
}
missing, unexpected = model.load_state_dict(backbone_state, strict=False)

# Reuse the learned previous classifier when its 768 -> 2 weights exist.
old_linear_w = state.get("classifier.2.weight")
old_linear_b = state.get("classifier.2.bias")

if old_linear_w is None:
    old_linear_w = state.get("classifier.3.weight")
    old_linear_b = state.get("classifier.3.bias")

if old_linear_w is not None and tuple(old_linear_w.shape) == tuple(model.classifier[2].fc.weight.shape):
    with torch.no_grad():
        model.classifier[2].fc.weight.copy_(old_linear_w)
        model.classifier[2].fc.bias.copy_(old_linear_b)
    print("✓ Reused pretrained previous classifier weights.")
else:
    print("! Old classifier shape not found; classifier remains randomly initialized.")

print("Loaded pretrained backbone from:", EXP6_CHECKPOINT)
print("Missing keys after backbone load:", len(missing))
print("Unexpected keys:", len(unexpected))
print(model.classifier)


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 152MB/s]


NameError: name 'USE_BATCHNORM_HEAD' is not defined

In [ ]:
# =========================
# 12. Progressive unfreezing policy
# =========================
# EXP-5 froze early layers and used very high LR.
# Here: all stages are trainable, but early layers move extremely slowly.

for p in model.parameters():
    p.requires_grad = True

model = model.to(device)

if device.type == "cuda":
    model = model.to(memory_format=torch.channels_last)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params: {total_params/1e6:.2f}M")
print(f"Trainable params: {trainable_params/1e6:.2f}M")
print(model.classifier)


In [ ]:
# =========================
# 13. EXP-7 moderate class-balanced loss
# =========================
class_counts = train_df["label"].value_counts().sort_index()
print("Class counts:", class_counts.to_dict())

# Explicit moderate weights. EXP-5's inverse-frequency weighting was too strong,
# so EXP-7 uses only a 1.5x penalty for REAL errors.
class_weights = torch.tensor(
    [REAL_CLASS_WEIGHT, FAKE_CLASS_WEIGHT],
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=LABEL_SMOOTHING
)

print("Class weights [REAL, FAKE]:", class_weights.tolist())
print(criterion)

In [ ]:
# =========================
# 14. Discriminative optimizer
# =========================
features = model.features

param_groups = [
    {
        "params": list(features[0].parameters()),
        "lr": LR_STAGE1,
        "name": "stage1"
    },
    {
        "params": list(features[1].parameters()) + list(features[2].parameters()),
        "lr": LR_STAGE2,
        "name": "stage2"
    },
    {
        "params": list(features[3].parameters()) + list(features[4].parameters()) + list(features[5].parameters()),
        "lr": LR_STAGE3,
        "name": "stage3"
    },
    {
        "params": list(features[6].parameters()) + list(features[7].parameters()) +
                  list(model.avgpool.parameters()) + list(model.classifier.parameters()),
        "lr": LR_STAGE4_HEAD,
        "name": "stage4+head"
    }
]

optimizer = torch.optim.AdamW(
    param_groups,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999)
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-7
)

for g in param_groups:
    n = sum(p.numel() for p in g["params"])
    print(f"{g['name']:12s} | LR={g['lr']:.1e} | Params={n/1e6:.2f}M")


In [ ]:
# =========================
# 15. Training / validation functions
# =========================
def move_images(images):
    images = images.to(device, non_blocking=True)
    if device.type == "cuda":
        images = images.contiguous(memory_format=torch.channels_last)
    return images

def train_one_epoch(model, loader):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, desc="Train", leave=False):
        images = move_images(images)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        all_preds.append(preds.detach().cpu())
        all_labels.append(labels.detach().cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()
    return running_loss / len(loader.dataset), f1_score(y_true, y_pred, pos_label=1, zero_division=0)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    running_loss = 0.0
    all_probs, all_labels = [], []

    for images, labels in tqdm(loader, desc="Eval", leave=False):
        images = move_images(images)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=1)[:, 1]
        running_loss += loss.item() * images.size(0)
        all_probs.append(probs.cpu())
        all_labels.append(labels.cpu())

    y_prob = torch.cat(all_probs).numpy()
    y_true = torch.cat(all_labels).numpy()
    y_pred = (y_prob >= 0.5).astype(int)

    return {
        "loss": running_loss / len(loader.dataset),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob
    }

def threshold_analysis(y_true, y_prob, start=0.05, end=0.80, step=0.01):
    rows = []
    for t in np.arange(start, end + 1e-9, step):
        pred = (y_prob >= t).astype(int)
        cm = confusion_matrix(y_true, pred, labels=[0,1])
        tn, fp, fn, tp = cm.ravel()
        real_recall = tn / (tn + fp) if (tn + fp) else 0.0
        fake_recall = tp / (tp + fn) if (tp + fn) else 0.0
        rows.append({
            "threshold": round(float(t), 3),
            "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall_fake": fake_recall,
            "recall_real": real_recall,
            "f1": f1_score(y_true, pred, zero_division=0),
            "balanced_accuracy": (real_recall + fake_recall) / 2
        })
    return pd.DataFrame(rows)

In [ ]:
# =========================
# 16. EXP-7 Training — TRAIN + VAL ONLY
# =========================
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

history = []
best_f1 = -1.0
best_auc = -1.0
best_acc = -1.0
best_epoch_f1 = -1
best_epoch_acc = -1
epochs_without_improvement = 0

EXP7_CHECKPOINT_F1 = os.path.join(
    CHECKPOINT_DIR, f"{MODEL_NAME}_{EXP_NAME}_best_f1.pth"
)
EXP7_CHECKPOINT_ACC = os.path.join(
    CHECKPOINT_DIR, f"{MODEL_NAME}_{EXP_NAME}_best_accuracy.pth"
)
EXP7_HISTORY = os.path.join(
    HISTORY_DIR, f"{MODEL_NAME}_{EXP_NAME}_history.csv"
)

print("DATA PROTOCOL: TRAIN + VAL ONLY")
print("TEST USED: False")

total_start = time.time()

for epoch in range(EPOCHS):
    t0 = time.time()
    print("\n" + "="*70)
    print(f"EXP-7 Epoch {epoch+1}/{EPOCHS}")

    train_loss, train_f1 = train_one_epoch(model, train_loader)
    val = evaluate(model, val_loader)
    scheduler.step()
    lrs = [g["lr"] for g in optimizer.param_groups]

    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_f1": train_f1,
        "val_loss": val["loss"],
        "val_accuracy": val["accuracy"],
        "val_precision": val["precision"],
        "val_recall_fake": val["recall"],
        "val_f1": val["f1"],
        "val_roc_auc": val["roc_auc"],
        "lr_stage1": lrs[0], "lr_stage2": lrs[1],
        "lr_stage3": lrs[2], "lr_stage4_head": lrs[3],
        "epoch_time_sec": time.time()-t0
    }
    history.append(row)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train F1   : {train_f1:.4f}")
    print(f"Val Loss   : {val['loss']:.4f}")
    print(f"Val Acc    : {val['accuracy']:.4f}")
    print(f"Val Prec   : {val['precision']:.4f}")
    print(f"Val Recall : {val['recall']:.4f}")
    print(f"Val F1     : {val['f1']:.4f}")
    print(f"Val AUC    : {val['roc_auc']:.4f}")
    print(f"Epoch time : {row['epoch_time_sec']:.1f}s")

    better_f1 = (val["f1"] > best_f1) or (
        val["f1"] == best_f1 and val["roc_auc"] > best_auc
    )
    if better_f1:
        best_f1, best_auc, best_epoch_f1 = val["f1"], val["roc_auc"], epoch + 1
        epochs_without_improvement = 0
        torch.save({
            "model_name": MODEL_NAME, "exp_name": EXP_NAME,
            "finetuned_from": EXP6_CHECKPOINT, "epoch": best_epoch_f1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if USE_AMP else None,
            "metrics": {k:v for k,v in val.items() if k not in ["y_true","y_pred","y_prob"]},
            "config": {
                "img_size": IMG_SIZE, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
                "lr_stage1": LR_STAGE1, "lr_stage2": LR_STAGE2,
                "lr_stage3": LR_STAGE3, "lr_stage4_head": LR_STAGE4_HEAD,
                "weight_decay": WEIGHT_DECAY, "label_smoothing": LABEL_SMOOTHING,
                "dropout": DROPOUT_P, "real_class_weight": REAL_CLASS_WEIGHT,
                "fake_class_weight": FAKE_CLASS_WEIGHT, "jpeg_p": JPEG_P,
                "blur_p": BLUR_P, "noise_p": NOISE_P, "color_p": COLOR_P,
                "rotation_deg": ROTATION_DEG, "seed": SEED, "test_used": False
            }
        }, EXP7_CHECKPOINT_F1)
        print("✓ BEST-F1 CHECKPOINT SAVED")
    else:
        epochs_without_improvement += 1

    if val["accuracy"] > best_acc:
        best_acc, best_epoch_acc = val["accuracy"], epoch + 1
        torch.save({
            "model_name": MODEL_NAME, "exp_name": EXP_NAME,
            "finetuned_from": EXP6_CHECKPOINT, "epoch": best_epoch_acc,
            "model_state_dict": model.state_dict(),
            "metrics": {k:v for k,v in val.items() if k not in ["y_true","y_pred","y_prob"]},
            "config": {
                "real_class_weight": REAL_CLASS_WEIGHT,
                "fake_class_weight": FAKE_CLASS_WEIGHT,
                "weight_decay": WEIGHT_DECAY, "dropout": DROPOUT_P,
                "label_smoothing": LABEL_SMOOTHING, "test_used": False
            }
        }, EXP7_CHECKPOINT_ACC)
        print("✓ BEST-ACCURACY CHECKPOINT SAVED")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping by validation F1.")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(EXP7_HISTORY, index=False)

print("\nEXP-7 finished.")
print("Best F1:", best_f1, "at epoch", best_epoch_f1)
print("Best Accuracy@0.5:", best_acc, "at epoch", best_epoch_acc)
print("Time:", (time.time()-total_start)/60, "min")

In [ ]:
# =========================
# 17. Learning curves — bias / variance
# =========================
h = pd.read_csv(EXP7_HISTORY)

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(h["epoch"], h["train_f1"], marker="o", label="Train F1")
ax.plot(h["epoch"], h["val_f1"], marker="o", label="Val F1")
ax.set_xlabel("Epoch"); ax.set_ylabel("F1")
ax.set_title("EXP-7 Train vs Validation F1")
ax.grid(True); ax.legend(); plt.show()

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(h["epoch"], h["train_loss"], marker="o", label="Train Loss")
ax.plot(h["epoch"], h["val_loss"], marker="o", label="Val Loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("EXP-7 Train vs Validation Loss")
ax.grid(True); ax.legend(); plt.show()

In [ ]:
# =========================
# 18. BEST EXP-7 — VALIDATION ONLY
# =========================
best_ckpt = torch.load(EXP7_CHECKPOINT_F1, map_location=device, weights_only=False)
model.load_state_dict(best_ckpt["model_state_dict"])
val_best = evaluate(model, val_loader)

print("BEST EXP-7 — VALIDATION ONLY")
for k in ["loss","accuracy","precision","recall","f1","roc_auc"]:
    print(f"{k:10s}: {val_best[k]:.4f}")

cm = confusion_matrix(val_best["y_true"], val_best["y_pred"], labels=[0,1])
tn, fp, fn, tp = cm.ravel()
print("\nConfusion Matrix [REAL, FAKE]:")
print(cm)
print(f"REAL recall: {tn/(tn+fp):.4f}")
print(f"FAKE recall: {tp/(tp+fn):.4f}")

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["REAL","FAKE"], yticklabels=["REAL","FAKE"])
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("EXP-7 Validation Confusion Matrix")
plt.show()

In [ ]:
# =========================
# 19. Threshold analysis — VALIDATION ONLY
# =========================
threshold_df = threshold_analysis(
    val_best["y_true"], val_best["y_prob"],
    start=0.05, end=0.80, step=0.01
)

best_acc_row = threshold_df.loc[threshold_df["accuracy"].idxmax()]
best_f1_row = threshold_df.loc[threshold_df["f1"].idxmax()]
best_bal_row = threshold_df.loc[threshold_df["balanced_accuracy"].idxmax()]

print("Best threshold by VAL Accuracy:")
print(best_acc_row)
print("\nBest threshold by VAL F1:")
print(best_f1_row)
print("\nBest threshold by VAL Balanced Accuracy:")
print(best_bal_row)

plt.figure(figsize=(10,5))
for col in ["accuracy","precision","recall_real","recall_fake","f1","balanced_accuracy"]:
    plt.plot(threshold_df["threshold"], threshold_df[col], label=col)
plt.axvline(0.50, linestyle="--", label="default 0.50")
plt.xlabel("Threshold"); plt.ylabel("Metric")
plt.title("EXP-7 Validation Threshold Analysis")
plt.legend(); plt.grid(True); plt.show()

selected_threshold = float(best_acc_row["threshold"])
print(f"Selected threshold for VAL analysis: {selected_threshold:.3f}")

In [ ]:
# =========================
# 20. Selected-threshold validation report
# =========================
pred_selected = (val_best["y_prob"] >= selected_threshold).astype(int)
cm_sel = confusion_matrix(val_best["y_true"], pred_selected, labels=[0,1])
tn, fp, fn, tp = cm_sel.ravel()

selected_metrics = {
    "threshold": selected_threshold,
    "accuracy": accuracy_score(val_best["y_true"], pred_selected),
    "precision": precision_score(val_best["y_true"], pred_selected, zero_division=0),
    "recall_fake": recall_score(val_best["y_true"], pred_selected, zero_division=0),
    "recall_real": tn/(tn+fp) if (tn+fp) else 0.0,
    "f1": f1_score(val_best["y_true"], pred_selected, zero_division=0),
    "balanced_accuracy": ((tn/(tn+fp)) + (tp/(tp+fn))) / 2
}

print("EXP-7 SELECTED THRESHOLD — VALIDATION ONLY")
for k,v in selected_metrics.items(): print(f"{k:20s}: {v:.4f}")
print("\nConfusion Matrix:")
print(cm_sel)

In [ ]:
# =========================
# 21. Save EXP-7 summary — NO TEST ACCESS
# =========================
summary = {
    "experiment": EXP_NAME,
    "model": MODEL_NAME,
    "source_checkpoint": EXP6_CHECKPOINT,
    "best_f1_checkpoint": EXP7_CHECKPOINT_F1,
    "best_accuracy_checkpoint": EXP7_CHECKPOINT_ACC,
    "best_f1_epoch": int(best_epoch_f1),
    "best_accuracy_epoch": int(best_epoch_acc),
    "val_metrics_at_0.5": {
        k: float(val_best[k]) for k in ["loss","accuracy","precision","recall","f1","roc_auc"]
    },
    "validation_selected_threshold": selected_threshold,
    "validation_selected_threshold_metrics": selected_metrics,
    "config": {
        "batch_size": BATCH_SIZE, "epochs_planned": EPOCHS,
        "lr_stage1": LR_STAGE1, "lr_stage2": LR_STAGE2,
        "lr_stage3": LR_STAGE3, "lr_stage4_head": LR_STAGE4_HEAD,
        "weight_decay": WEIGHT_DECAY, "label_smoothing": LABEL_SMOOTHING,
        "dropout": DROPOUT_P, "real_class_weight": REAL_CLASS_WEIGHT,
        "fake_class_weight": FAKE_CLASS_WEIGHT, "jpeg_p": JPEG_P,
        "blur_p": BLUR_P, "noise_p": NOISE_P, "color_p": COLOR_P,
        "rotation_deg": ROTATION_DEG, "seed": SEED, "test_used": False
    }
}

summary_path = os.path.join(RESULT_DIR, f"{MODEL_NAME}_{EXP_NAME}_summary.json")
with open(summary_path, "w") as f: json.dump(summary, f, indent=2)

print("Saved:", summary_path)
print("==============================")
print("EXP-7 COMPLETE")
print("TEST USED: FALSE")
print("==============================")

## Cách quyết định sau EXP-7

So sánh với EXP-6 theo thứ tự:

1. Validation Accuracy tăng.
2. REAL recall tăng rõ rệt (EXP-6 khoảng 45.6%).
3. F1 không giảm đáng kể.
4. ROC-AUC không giảm.
5. Train–Val gap nhỏ hơn.

**Nếu Accuracy + REAL recall cùng tăng:** giữ EXP-7 làm ứng viên tốt nhất.

**Nếu REAL recall tăng nhưng F1/AUC giảm mạnh:** EXP-7 đang weight REAL hơi mạnh → EXP-8 có thể giảm REAL weight xuống 1.25.

**Nếu train F1 vẫn rất cao nhưng val F1/accuracy không tăng:** không tiếp tục tăng LR; cần kiểm tra frame quality, label noise và domain gap.

> **TEST vẫn khóa hoàn toàn.** Chỉ mở TEST sau khi chốt xong model + threshold bằng validation.